# Model Development

In [39]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [40]:
data = pd.read_csv(
    "../Data/Processed/solar_model_data.csv"
)

data["time_utc"] = pd.to_datetime(
    data["time_utc"],
    utc=True
)

data["time_sl"] = (
    pd.to_datetime(
        data["time_sl"],
        utc=True
    )
    .dt.tz_convert("Asia/Colombo")
)

print("Dataset shape:", data.shape)

Dataset shape: (227218, 21)


Sort Dataset Chronologically

In [41]:
data = data.sort_values(
    ["time_utc", "district"]
).reset_index(drop=True)

print(
    "First timestamp:",
    data["time_utc"].min()
)

print(
    "Last timestamp:",
    data["time_utc"].max()
)

First timestamp: 2022-01-01 02:00:00+00:00
Last timestamp: 2023-12-31 13:00:00+00:00


 Create Chronological Train Validation Test Split

In [42]:
unique_times = (
    data["time_utc"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

train_end = int(
    len(unique_times) * 0.70
)

validation_end = int(
    len(unique_times) * 0.85
)

train_last_time = unique_times.iloc[
    train_end - 1
]

validation_last_time = unique_times.iloc[
    validation_end - 1
]

print(
    "Training ends:",
    train_last_time
)

print(
    "Validation ends:",
    validation_last_time
)

Training ends: 2023-06-03 03:00:00+00:00
Validation ends: 2023-09-17 01:00:00+00:00


In [43]:
train_data = data[
    data["time_utc"] <= train_last_time
].copy()

validation_data = data[
    (data["time_utc"] > train_last_time)
    &
    (
        data["time_utc"]
        <= validation_last_time
    )
].copy()

test_data = data[
    data["time_utc"] > validation_last_time
].copy()

Dataset Split

In [44]:
print(
    "Training rows:",
    len(train_data)
)

print(
    "Validation rows:",
    len(validation_data)
)

print(
    "Test rows:",
    len(test_data)
)

print()

print(
    "Training percentage:",
    round(
        len(train_data) / len(data) * 100,
        2
    )
)

print(
    "Validation percentage:",
    round(
        len(validation_data) / len(data) * 100,
        2
    )
)

print(
    "Test percentage:",
    round(
        len(test_data) / len(data) * 100,
        2
    )
)

Training rows: 159313
Validation rows: 34383
Test rows: 33522

Training percentage: 70.11
Validation percentage: 15.13
Test percentage: 14.75


In [45]:
print(
    "Train:",
    train_data["time_sl"].min(),
    "to",
    train_data["time_sl"].max()
)

print(
    "Validation:",
    validation_data["time_sl"].min(),
    "to",
    validation_data["time_sl"].max()
)

print(
    "Test:",
    test_data["time_sl"].min(),
    "to",
    test_data["time_sl"].max()
)

print()

print(
    "Train districts:",
    train_data["district"].nunique()
)

print(
    "Validation districts:",
    validation_data["district"].nunique()
)

print(
    "Test districts:",
    test_data["district"].nunique()
)

Train: 2022-01-01 07:30:00+05:30 to 2023-06-03 08:30:00+05:30
Validation: 2023-06-03 09:30:00+05:30 to 2023-09-17 06:30:00+05:30
Test: 2023-09-17 07:30:00+05:30 to 2023-12-31 18:30:00+05:30

Train districts: 25
Validation districts: 25
Test districts: 25


In [46]:
print(
    "Chronological split valid:",
    (
        train_data["time_utc"].max()
        <
        validation_data["time_utc"].min()
        <
        test_data["time_utc"].min()
    )
)

Chronological split valid: True


Define Features and Target

In [47]:
feature_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "cloud_cover",
    "wind_speed_10m",
    "global_tilted_irradiance",
    "diffuse_radiation",
    "sunshine_duration",
    "latitude",
    "longitude",
    "elevation",
    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "gti_lag_1",
    "cloud_cover_lag_1"
]

target_column = "P"

In [48]:
X_train = train_data[feature_columns]
y_train = train_data[target_column]

X_validation = validation_data[feature_columns]
y_validation = validation_data[target_column]

X_test = test_data[feature_columns]
y_test = test_data[target_column]

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

X_train: (159313, 17)
X_validation: (34383, 17)
X_test: (33522, 17)


Train Naive Baseline

In [49]:
baseline_model = DummyRegressor(
    strategy="mean"
)

baseline_model.fit(
    X_train,
    y_train
)

baseline_predictions = (
    baseline_model.predict(
        X_validation
    )
)

In [50]:
baseline_mae = mean_absolute_error(
    y_validation,
    baseline_predictions
)

baseline_mse = mean_squared_error(
    y_validation,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    baseline_mse
)

baseline_r2 = r2_score(
    y_validation,
    baseline_predictions
)

print("Naive Baseline")
print("MAE:", round(baseline_mae, 3))
print("MSE:", round(baseline_mse, 3))
print("RMSE:", round(baseline_rmse, 3))
print("R²:", round(baseline_r2, 4))

Naive Baseline
MAE: 194.74
MSE: 48495.318
RMSE: 220.217
R²: -0.003


Calculate Irradiance Baseline

In [51]:
irradiance_predictions = (
    X_validation["global_tilted_irradiance"]
    * 0.86
)

irradiance_mae = mean_absolute_error(
    y_validation,
    irradiance_predictions
)

irradiance_mse = mean_squared_error(
    y_validation,
    irradiance_predictions
)

irradiance_rmse = np.sqrt(
    irradiance_mse
)

irradiance_r2 = r2_score(
    y_validation,
    irradiance_predictions
)

print("Irradiance Baseline")
print("MAE:", round(irradiance_mae, 3))
print("MSE:", round(irradiance_mse, 3))
print("RMSE:", round(irradiance_rmse, 3))
print("R²:", round(irradiance_r2, 4))

Irradiance Baseline
MAE: 95.959
MSE: 16482.894
RMSE: 128.386
R²: 0.6591


 Linear Regression

In [52]:
linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

linear_predictions = (
    linear_model.predict(
        X_validation
    )
)

In [53]:
linear_mae = mean_absolute_error(
    y_validation,
    linear_predictions
)

linear_mse = mean_squared_error(
    y_validation,
    linear_predictions
)

linear_rmse = np.sqrt(
    linear_mse
)

linear_r2 = r2_score(
    y_validation,
    linear_predictions
)

print("Linear Regression")
print("MAE:", round(linear_mae, 3))
print("MSE:", round(linear_mse, 3))
print("RMSE:", round(linear_rmse, 3))
print("R²:", round(linear_r2, 4))

Linear Regression
MAE: 45.953
MSE: 4258.447
RMSE: 65.257
R²: 0.9119


Initial Model Performance

In [54]:
initial_results = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Irradiance Baseline",
        "Linear Regression"
    ],
    "MAE": [
        baseline_mae,
        irradiance_mae,
        linear_mae
    ],
    "MSE": [
        baseline_mse,
        irradiance_mse,
        linear_mse
    ],
    "RMSE": [
        baseline_rmse,
        irradiance_rmse,
        linear_rmse
    ],
    "R2": [
        baseline_r2,
        irradiance_r2,
        linear_r2
    ]
})

initial_results.round(4)

,Model,MAE,MSE,RMSE,R2
0,Naive Baseline,194.7398,48495.3177,220.2165,-0.0030
1,Irradiance Baseline,95.9590,16482.8938,128.3857,0.6591
2,Linear Regression,45.9533,4258.4467,65.2568,0.9119


Check Linear Regression Predictions

In [55]:
print(
    "Negative predictions:",
    (linear_predictions < 0).sum()
)

print(
    "Minimum prediction:",
    linear_predictions.min()
)

print(
    "Maximum prediction:",
    linear_predictions.max()
)

Negative predictions: 2890
Minimum prediction: -138.4727862568891
Maximum prediction: 685.4415047902185


 Non-Negative Linear Regression Predictions

In [56]:
linear_predictions_clipped = np.clip(
    linear_predictions,
    0,
    None
)

clipped_mae = mean_absolute_error(
    y_validation,
    linear_predictions_clipped
)

clipped_mse = mean_squared_error(
    y_validation,
    linear_predictions_clipped
)

clipped_rmse = np.sqrt(
    clipped_mse
)

clipped_r2 = r2_score(
    y_validation,
    linear_predictions_clipped
)

print("Linear Regression - Clipped")
print("MAE:", round(clipped_mae, 3))
print("MSE:", round(clipped_mse, 3))
print("RMSE:", round(clipped_rmse, 3))
print("R²:", round(clipped_r2, 4))

Linear Regression - Clipped
MAE: 44.13
MSE: 4172.53
RMSE: 64.595
R²: 0.9137


In [57]:
linear_comparison = pd.DataFrame({
    "Version": [
        "Original Linear Regression",
        "Non-Negative Linear Regression"
    ],
    "MAE": [
        linear_mae,
        clipped_mae
    ],
    "MSE": [
        linear_mse,
        clipped_mse
    ],
    "RMSE": [
        linear_rmse,
        clipped_rmse
    ],
    "R2": [
        linear_r2,
        clipped_r2
    ]
})

linear_comparison.round(4)

,Version,MAE,MSE,RMSE,R2
0,Original Linear Regression,45.9533,4258.4467,65.2568,0.9119
1,Non-Negative Linear Regression,44.1300,4172.5299,64.5951,0.9137


Train Random Forest

In [58]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [59]:
rf_predictions = random_forest_model.predict(
    X_validation
)

In [60]:
rf_mae = mean_absolute_error(
    y_validation,
    rf_predictions
)

rf_mse = mean_squared_error(
    y_validation,
    rf_predictions
)

rf_rmse = np.sqrt(
    rf_mse
)

rf_r2 = r2_score(
    y_validation,
    rf_predictions
)

print("Random Forest")
print("MAE:", round(rf_mae, 3))
print("MSE:", round(rf_mse, 3))
print("RMSE:", round(rf_rmse, 3))
print("R²:", round(rf_r2, 4))

Random Forest
MAE: 39.212
MSE: 3695.872
RMSE: 60.794
R²: 0.9236


Random Forest Training Performance

In [61]:
rf_train_predictions = (
    random_forest_model.predict(
        X_train
    )
)

rf_train_mae = mean_absolute_error(
    y_train,
    rf_train_predictions
)

rf_train_mse = mean_squared_error(
    y_train,
    rf_train_predictions
)

rf_train_rmse = np.sqrt(
    rf_train_mse
)

rf_train_r2 = r2_score(
    y_train,
    rf_train_predictions
)

print("Random Forest - Training")
print("MAE:", round(rf_train_mae, 3))
print("MSE:", round(rf_train_mse, 3))
print("RMSE:", round(rf_train_rmse, 3))
print("R²:", round(rf_train_r2, 4))

Random Forest - Training
MAE: 14.644
MSE: 518.959
RMSE: 22.781
R²: 0.9898


Random Forest Predictions

In [62]:
print(
    "Negative predictions:",
    (rf_predictions < 0).sum()
)

print(
    "Minimum prediction:",
    rf_predictions.min()
)

print(
    "Maximum prediction:",
    rf_predictions.max()
)

Negative predictions: 0
Minimum prediction: 0.0
Maximum prediction: 712.3722999999997


Compare Model Performance

In [63]:
model_results = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Irradiance Baseline",
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        baseline_mae,
        irradiance_mae,
        linear_mae,
        rf_mae
    ],
    "MSE": [
        baseline_mse,
        irradiance_mse,
        linear_mse,
        rf_mse
    ],
    "RMSE": [
        baseline_rmse,
        irradiance_rmse,
        linear_rmse,
        rf_rmse
    ],
    "R2": [
        baseline_r2,
        irradiance_r2,
        linear_r2,
        rf_r2
    ]
})

model_results.round(4)

,Model,MAE,MSE,RMSE,R2
0,Naive Baseline,194.7398,48495.3177,220.2165,-0.0030
1,Irradiance Baseline,95.9590,16482.8938,128.3857,0.6591
2,Linear Regression,45.9533,4258.4467,65.2568,0.9119
3,Random Forest,39.2123,3695.8725,60.7937,0.9236


# XGBoost

In [64]:
xgboost_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgboost_model.fit(
    X_train,
    y_train
)

xgb_predictions = xgboost_model.predict(
    X_validation
)

Evaluate XGBoost

In [65]:
xgb_mae = mean_absolute_error(
    y_validation,
    xgb_predictions
)

xgb_mse = mean_squared_error(
    y_validation,
    xgb_predictions
)

xgb_rmse = np.sqrt(
    xgb_mse
)

xgb_r2 = r2_score(
    y_validation,
    xgb_predictions
)

print("XGBoost")
print("MAE:", round(xgb_mae, 3))
print("MSE:", round(xgb_mse, 3))
print("RMSE:", round(xgb_rmse, 3))
print("R²:", round(xgb_r2, 4))

XGBoost
MAE: 42.643
MSE: 4165.082
RMSE: 64.537
R²: 0.9139


In [66]:
print(
    "Negative predictions:",
    (xgb_predictions < 0).sum()
)

print(
    "Minimum prediction:",
    xgb_predictions.min()
)

print(
    "Maximum prediction:",
    xgb_predictions.max()
)

Negative predictions: 1486
Minimum prediction: -40.495235
Maximum prediction: 715.7352


XGBoost Training Performance

In [67]:
xgb_train_predictions = xgboost_model.predict(
    X_train
)

xgb_train_mae = mean_absolute_error(
    y_train,
    xgb_train_predictions
)

xgb_train_mse = mean_squared_error(
    y_train,
    xgb_train_predictions
)

xgb_train_rmse = np.sqrt(
    xgb_train_mse
)

xgb_train_r2 = r2_score(
    y_train,
    xgb_train_predictions
)

print("XGBoost - Training")
print("MAE:", round(xgb_train_mae, 3))
print("MSE:", round(xgb_train_mse, 3))
print("RMSE:", round(xgb_train_rmse, 3))
print("R²:", round(xgb_train_r2, 4))

XGBoost - Training
MAE: 35.08
MSE: 2708.025
RMSE: 52.039
R²: 0.9469


# LightGBM

In [68]:
lightgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lightgbm_model.fit(
    X_train,
    y_train
)

lgb_predictions = lightgbm_model.predict(
    X_validation
)

In [69]:
lgb_mae = mean_absolute_error(
    y_validation,
    lgb_predictions
)

lgb_mse = mean_squared_error(
    y_validation,
    lgb_predictions
)

lgb_rmse = np.sqrt(
    lgb_mse
)

lgb_r2 = r2_score(
    y_validation,
    lgb_predictions
)

print("LightGBM")
print("MAE:", round(lgb_mae, 3))
print("MSE:", round(lgb_mse, 3))
print("RMSE:", round(lgb_rmse, 3))
print("R²:", round(lgb_r2, 4))

LightGBM
MAE: 42.407
MSE: 4203.88
RMSE: 64.837
R²: 0.9131


In [70]:
print(
    "Negative predictions:",
    (lgb_predictions < 0).sum()
)

print(
    "Minimum prediction:",
    lgb_predictions.min()
)

print(
    "Maximum prediction:",
    lgb_predictions.max()
)

Negative predictions: 1579
Minimum prediction: -29.66697623714569
Maximum prediction: 705.5385011270587


# CatBoost

In [71]:
catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

catboost_model.fit(
    X_train,
    y_train
)

cat_predictions = catboost_model.predict(
    X_validation
)

In [72]:
cat_mae = mean_absolute_error(
    y_validation,
    cat_predictions
)

cat_mse = mean_squared_error(
    y_validation,
    cat_predictions
)

cat_rmse = np.sqrt(
    cat_mse
)

cat_r2 = r2_score(
    y_validation,
    cat_predictions
)

print("CatBoost")
print("MAE:", round(cat_mae, 3))
print("MSE:", round(cat_mse, 3))
print("RMSE:", round(cat_rmse, 3))
print("R²:", round(cat_r2, 4))

CatBoost
MAE: 40.975
MSE: 3849.755
RMSE: 62.046
R²: 0.9204


In [73]:
print(
    "Negative predictions:",
    (cat_predictions < 0).sum()
)

print(
    "Minimum prediction:",
    cat_predictions.min()
)

print(
    "Maximum prediction:",
    cat_predictions.max()
)

Negative predictions: 1794
Minimum prediction: -29.085201265405885
Maximum prediction: 711.7150979590955


LightGBM Training Performance

In [77]:
lgb_train_predictions = lightgbm_model.predict(
    X_train
)

lgb_train_mae = mean_absolute_error(
    y_train,
    lgb_train_predictions
)

lgb_train_mse = mean_squared_error(
    y_train,
    lgb_train_predictions
)

lgb_train_rmse = np.sqrt(
    lgb_train_mse
)

lgb_train_r2 = r2_score(
    y_train,
    lgb_train_predictions
)

print("LightGBM - Training")
print("MAE:", round(lgb_train_mae, 3))
print("MSE:", round(lgb_train_mse, 3))
print("RMSE:", round(lgb_train_rmse, 3))
print("R²:", round(lgb_train_r2, 4))

LightGBM - Training
MAE: 36.363
MSE: 2860.992
RMSE: 53.488
R²: 0.9439


CatBoost Training Performance

In [78]:
cat_train_predictions = catboost_model.predict(
    X_train
)

cat_train_mae = mean_absolute_error(
    y_train,
    cat_train_predictions
)

cat_train_mse = mean_squared_error(
    y_train,
    cat_train_predictions
)

cat_train_rmse = np.sqrt(
    cat_train_mse
)

cat_train_r2 = r2_score(
    y_train,
    cat_train_predictions
)

print("CatBoost - Training")
print("MAE:", round(cat_train_mae, 3))
print("MSE:", round(cat_train_mse, 3))
print("RMSE:", round(cat_train_rmse, 3))
print("R²:", round(cat_train_r2, 4))

CatBoost - Training
MAE: 40.622
MSE: 3692.45
RMSE: 60.766
R²: 0.9276


# All Model Evaluation

In [79]:
all_model_results = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Irradiance Baseline",
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "MAE": [
        baseline_mae,
        irradiance_mae,
        linear_mae,
        rf_mae,
        xgb_mae,
        lgb_mae,
        cat_mae
    ],
    "MSE": [
        baseline_mse,
        irradiance_mse,
        linear_mse,
        rf_mse,
        xgb_mse,
        lgb_mse,
        cat_mse
    ],
    "RMSE": [
        baseline_rmse,
        irradiance_rmse,
        linear_rmse,
        rf_rmse,
        xgb_rmse,
        lgb_rmse,
        cat_rmse
    ],
    "R2": [
        baseline_r2,
        irradiance_r2,
        linear_r2,
        rf_r2,
        xgb_r2,
        lgb_r2,
        cat_r2
    ]
})

all_model_results.sort_values(
    "RMSE"
).round(4)

,Model,MAE,MSE,RMSE,R2
3,Random Forest,39.2123,3695.8725,60.7937,0.9236
6,CatBoost,40.9747,3849.7547,62.0464,0.9204
4,XGBoost,42.6434,4165.0816,64.5374,0.9139
5,LightGBM,42.4067,4203.8799,64.8373,0.9131
2,Linear Regression,45.9533,4258.4467,65.2568,0.9119
1,Irradiance Baseline,95.9590,16482.8938,128.3857,0.6591
0,Naive Baseline,194.7398,48495.3177,220.2165,-0.0030


Tuning Random Forest

In [80]:
rf_parameters = [
    {
        "max_depth": 12,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "max_depth": 16,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "max_depth": 16,
        "min_samples_leaf": 4,
        "max_features": 1.0
    },
    {
        "max_depth": 20,
        "min_samples_leaf": 4,
        "max_features": 1.0
    },
    {
        "max_depth": 16,
        "min_samples_leaf": 2,
        "max_features": 0.8
    }
]

In [81]:
rf_tuning_results = []

for i, params in enumerate(
    rf_parameters,
    start=1
):
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_validation
    )

    mae = mean_absolute_error(
        y_validation,
        predictions
    )

    mse = mean_squared_error(
        y_validation,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_validation,
        predictions
    )

    rf_tuning_results.append({
        "Experiment": i,
        "Max Depth": params["max_depth"],
        "Min Samples Leaf": params["min_samples_leaf"],
        "Max Features": params["max_features"],
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

Compare Random Forest Tuning Results

In [82]:
rf_tuning_results_df = pd.DataFrame(
    rf_tuning_results
)

rf_tuning_results_df.sort_values(
    "RMSE"
).round(4)

,Experiment,Max Depth,Min Samples Leaf,Max Features,MAE,MSE,RMSE,R2
5,6,16,2,0.8,38.2887,3544.4250,59.5351,0.9267
3,4,16,4,1.0,38.4923,3597.9094,59.9826,0.9256
1,2,16,2,1.0,38.5756,3608.1446,60.0678,0.9254
0,1,12,2,1.0,38.8034,3610.6341,60.0886,0.9253
4,5,20,4,1.0,38.5741,3615.9438,60.1327,0.9252
2,3,20,2,1.0,38.7576,3635.1577,60.2923,0.9248


In [83]:
best_rf_result = (
    rf_tuning_results_df
    .sort_values("RMSE")
    .iloc[0]
)

best_rf_result

Experiment             6.000000
Max Depth             16.000000
Min Samples Leaf       2.000000
Max Features           0.800000
MAE                   38.288712
MSE                 3544.425010
RMSE                  59.535074
R2                     0.926691
Name: 5, dtype: float64

Compare Initial and Tuned Random Forest

In [84]:
rf_before_after = pd.DataFrame({
    "Model": [
        "Initial Random Forest",
        "Best Tuned Random Forest"
    ],
    "MAE": [
        rf_mae,
        best_rf_result["MAE"]
    ],
    "MSE": [
        rf_mse,
        best_rf_result["MSE"]
    ],
    "RMSE": [
        rf_rmse,
        best_rf_result["RMSE"]
    ],
    "R2": [
        rf_r2,
        best_rf_result["R2"]
    ]
})

rf_before_after.round(4)

,Model,MAE,MSE,RMSE,R2
0,Initial Random Forest,39.2123,3695.8725,60.7937,0.9236
1,Best Tuned Random Forest,38.2887,3544.4250,59.5351,0.9267


Final Tuned Random Forest

In [85]:
tuned_rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

tuned_rf_model.fit(
    X_train,
    y_train
)

tuned_rf_predictions = tuned_rf_model.predict(
    X_validation
)

Tuned Random Forest Training Performance

In [86]:
tuned_rf_train_predictions = tuned_rf_model.predict(
    X_train
)

tuned_rf_train_mae = mean_absolute_error(
    y_train,
    tuned_rf_train_predictions
)

tuned_rf_train_mse = mean_squared_error(
    y_train,
    tuned_rf_train_predictions
)

tuned_rf_train_rmse = np.sqrt(
    tuned_rf_train_mse
)

tuned_rf_train_r2 = r2_score(
    y_train,
    tuned_rf_train_predictions
)

print("Tuned Random Forest - Training")
print("MAE:", round(tuned_rf_train_mae, 3))
print("MSE:", round(tuned_rf_train_mse, 3))
print("RMSE:", round(tuned_rf_train_rmse, 3))
print("R²:", round(tuned_rf_train_r2, 4))

Tuned Random Forest - Training
MAE: 28.67
MSE: 1908.123
RMSE: 43.682
R²: 0.9626


Random Forest Training and Validation Comparison

In [87]:
rf_train_validation_comparison = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation"
    ],
    "MAE": [
        tuned_rf_train_mae,
        best_rf_result["MAE"]
    ],
    "MSE": [
        tuned_rf_train_mse,
        best_rf_result["MSE"]
    ],
    "RMSE": [
        tuned_rf_train_rmse,
        best_rf_result["RMSE"]
    ],
    "R2": [
        tuned_rf_train_r2,
        best_rf_result["R2"]
    ]
})

rf_train_validation_comparison.round(4)

,Dataset,MAE,MSE,RMSE,R2
0,Training,28.6699,1908.1228,43.6821,0.9626
1,Validation,38.2887,3544.4250,59.5351,0.9267


Tune CatBoost

In [88]:
cat_parameters = [
    {
        "iterations": 500,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 3
    },
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 5,
        "l2_leaf_reg": 3
    },
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 7,
        "l2_leaf_reg": 3
    },
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5
    },
    {
        "iterations": 400,
        "learning_rate": 0.08,
        "depth": 6,
        "l2_leaf_reg": 5
    }
]

In [89]:
cat_tuning_results = []

for i, params in enumerate(
    cat_parameters,
    start=1
):
    print("Running experiment", i)

    cat_tuning_model = CatBoostRegressor(
        iterations=params["iterations"],
        learning_rate=params["learning_rate"],
        depth=params["depth"],
        l2_leaf_reg=params["l2_leaf_reg"],
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    cat_tuning_model.fit(
        X_train,
        y_train
    )

    predictions = cat_tuning_model.predict(
        X_validation
    )

    mae = mean_absolute_error(
        y_validation,
        predictions
    )

    mse = mean_squared_error(
        y_validation,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_validation,
        predictions
    )

    cat_tuning_results.append({
        "Experiment": i,
        "Iterations": params["iterations"],
        "Learning Rate": params["learning_rate"],
        "Depth": params["depth"],
        "L2 Leaf Reg": params["l2_leaf_reg"],
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("RMSE:", round(rmse, 3))

Running experiment 1
RMSE: 61.074
Running experiment 2
RMSE: 61.838
Running experiment 3
RMSE: 62.717
Running experiment 4
RMSE: 61.917
Running experiment 5
RMSE: 63.022


Compare CatBoost Tuning Results

In [90]:
cat_tuning_results_df = pd.DataFrame(
    cat_tuning_results
)

cat_tuning_results_df.sort_values(
    "RMSE"
).round(4)

,Experiment,Iterations,Learning Rate,Depth,L2 Leaf Reg,MAE,MSE,RMSE,R2
0,1,500,0.03,6,3,40.2052,3729.9972,61.0737,0.9229
1,2,500,0.05,5,3,40.9441,3823.9493,61.8381,0.9209
3,4,500,0.05,6,5,40.8278,3833.6635,61.9166,0.9207
2,3,500,0.05,7,3,41.2332,3933.4593,62.7173,0.9186
4,5,400,0.08,6,5,41.5485,3971.7976,63.0222,0.9179


In [91]:
best_cat_result = (
    cat_tuning_results_df
    .sort_values("RMSE")
    .iloc[0]
)

best_cat_result

Experiment          1.000000
Iterations        500.000000
Learning Rate       0.030000
Depth               6.000000
L2 Leaf Reg         3.000000
MAE                40.205233
MSE              3729.997167
RMSE               61.073703
R2                  0.922852
Name: 0, dtype: float64

Compare Initial and Tuned CatBoost

In [92]:
cat_before_after = pd.DataFrame({
    "Model": [
        "Initial CatBoost",
        "Best Tuned CatBoost"
    ],
    "MAE": [
        cat_mae,
        best_cat_result["MAE"]
    ],
    "MSE": [
        cat_mse,
        best_cat_result["MSE"]
    ],
    "RMSE": [
        cat_rmse,
        best_cat_result["RMSE"]
    ],
    "R2": [
        cat_r2,
        best_cat_result["R2"]
    ]
})

cat_before_after.round(4)

,Model,MAE,MSE,RMSE,R2
0,Initial CatBoost,40.9747,3849.7547,62.0464,0.9204
1,Best Tuned CatBoost,40.2052,3729.9972,61.0737,0.9229


Final Tuned CatBoost

In [93]:
tuned_cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

tuned_cat_model.fit(
    X_train,
    y_train
)

tuned_cat_predictions = tuned_cat_model.predict(
    X_validation
)

Tuned CatBoost Training Performance

In [94]:
tuned_cat_train_predictions = tuned_cat_model.predict(
    X_train
)

tuned_cat_train_mae = mean_absolute_error(
    y_train,
    tuned_cat_train_predictions
)

tuned_cat_train_mse = mean_squared_error(
    y_train,
    tuned_cat_train_predictions
)

tuned_cat_train_rmse = np.sqrt(
    tuned_cat_train_mse
)

tuned_cat_train_r2 = r2_score(
    y_train,
    tuned_cat_train_predictions
)

print("Tuned CatBoost - Training")
print("MAE:", round(tuned_cat_train_mae, 3))
print("MSE:", round(tuned_cat_train_mse, 3))
print("RMSE:", round(tuned_cat_train_rmse, 3))
print("R²:", round(tuned_cat_train_r2, 4))

Tuned CatBoost - Training
MAE: 42.633
MSE: 4094.554
RMSE: 63.989
R²: 0.9197


CatBoost Training and Validation Comparison

In [95]:
cat_train_validation_comparison = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation"
    ],
    "MAE": [
        tuned_cat_train_mae,
        best_cat_result["MAE"]
    ],
    "MSE": [
        tuned_cat_train_mse,
        best_cat_result["MSE"]
    ],
    "RMSE": [
        tuned_cat_train_rmse,
        best_cat_result["RMSE"]
    ],
    "R2": [
        tuned_cat_train_r2,
        best_cat_result["R2"]
    ]
})

cat_train_validation_comparison.round(4)

,Dataset,MAE,MSE,RMSE,R2
0,Training,42.6329,4094.5541,63.9887,0.9197
1,Validation,40.2052,3729.9972,61.0737,0.9229


Compare Tuned Models

In [96]:
tuned_model_results = pd.DataFrame({
    "Model": [
        "Tuned Random Forest",
        "Tuned CatBoost"
    ],
    "MAE": [
        best_rf_result["MAE"],
        best_cat_result["MAE"]
    ],
    "MSE": [
        best_rf_result["MSE"],
        best_cat_result["MSE"]
    ],
    "RMSE": [
        best_rf_result["RMSE"],
        best_cat_result["RMSE"]
    ],
    "R2": [
        best_rf_result["R2"],
        best_cat_result["R2"]
    ]
})

tuned_model_results.sort_values(
    "RMSE"
).round(4)

,Model,MAE,MSE,RMSE,R2
0,Tuned Random Forest,38.2887,3544.4250,59.5351,0.9267
1,Tuned CatBoost,40.2052,3729.9972,61.0737,0.9229


 Tune XGBoost

In [97]:
xgb_parameters = [
    {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 7,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 700,
        "learning_rate": 0.03,
        "max_depth": 5,
        "subsample": 0.9,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 400,
        "learning_rate": 0.08,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.9
    }
]

In [98]:
xgb_tuning_results = []

for i, params in enumerate(
    xgb_parameters,
    start=1
):
    print("Running experiment", i)

    xgb_tuning_model = XGBRegressor(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        random_state=42,
        n_jobs=-1
    )

    xgb_tuning_model.fit(
        X_train,
        y_train
    )

    predictions = xgb_tuning_model.predict(
        X_validation
    )

    mae = mean_absolute_error(
        y_validation,
        predictions
    )

    mse = mean_squared_error(
        y_validation,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_validation,
        predictions
    )

    xgb_tuning_results.append({
        "Experiment": i,
        "Estimators": params["n_estimators"],
        "Learning Rate": params["learning_rate"],
        "Max Depth": params["max_depth"],
        "Subsample": params["subsample"],
        "Colsample": params["colsample_bytree"],
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("RMSE:", round(rmse, 3))

Running experiment 1
RMSE: 62.229
Running experiment 2
RMSE: 63.46
Running experiment 3
RMSE: 65.635
Running experiment 4
RMSE: 62.609
Running experiment 5
RMSE: 64.457


Compare XGBoost Tuning Results

In [99]:
xgb_tuning_results_df = pd.DataFrame(
    xgb_tuning_results
)

xgb_tuning_results_df.sort_values(
    "RMSE"
).round(4)

,Experiment,Estimators,Learning Rate,Max Depth,Subsample,Colsample,MAE,MSE,RMSE,R2
0,1,500,0.03,6,0.8,0.8,41.0639,3872.4644,62.2291,0.9199
3,4,700,0.03,5,0.9,0.8,41.6857,3919.8560,62.6088,0.9189
1,2,500,0.05,5,0.8,0.8,42.3194,4027.2048,63.4603,0.9167
4,5,400,0.08,5,0.8,0.9,43.0206,4154.7498,64.4573,0.9141
2,3,500,0.05,7,0.8,0.8,43.0831,4307.8998,65.6346,0.9109


In [100]:
best_xgb_result = (
    xgb_tuning_results_df
    .sort_values("RMSE")
    .iloc[0]
)

best_xgb_result

Experiment          1.000000
Estimators        500.000000
Learning Rate       0.030000
Max Depth           6.000000
Subsample           0.800000
Colsample           0.800000
MAE                41.063886
MSE              3872.464376
RMSE               62.229128
R2                  0.919906
Name: 0, dtype: float64

Compare Initial and Tuned XGBoost

In [101]:
xgb_before_after = pd.DataFrame({
    "Model": [
        "Initial XGBoost",
        "Best Tuned XGBoost"
    ],
    "MAE": [
        xgb_mae,
        best_xgb_result["MAE"]
    ],
    "MSE": [
        xgb_mse,
        best_xgb_result["MSE"]
    ],
    "RMSE": [
        xgb_rmse,
        best_xgb_result["RMSE"]
    ],
    "R2": [
        xgb_r2,
        best_xgb_result["R2"]
    ]
})

xgb_before_after.round(4)

,Model,MAE,MSE,RMSE,R2
0,Initial XGBoost,42.6434,4165.0816,64.5374,0.9139
1,Best Tuned XGBoost,41.0639,3872.4644,62.2291,0.9199


Final Tuned XGBoost

In [102]:
tuned_xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

tuned_xgb_model.fit(
    X_train,
    y_train
)

tuned_xgb_predictions = tuned_xgb_model.predict(
    X_validation
)

Tuned XGBoost Training Performance

In [103]:
tuned_xgb_train_predictions = tuned_xgb_model.predict(
    X_train
)

tuned_xgb_train_mae = mean_absolute_error(
    y_train,
    tuned_xgb_train_predictions
)

tuned_xgb_train_mse = mean_squared_error(
    y_train,
    tuned_xgb_train_predictions
)

tuned_xgb_train_rmse = np.sqrt(
    tuned_xgb_train_mse
)

tuned_xgb_train_r2 = r2_score(
    y_train,
    tuned_xgb_train_predictions
)

print("Tuned XGBoost - Training")
print("MAE:", round(tuned_xgb_train_mae, 3))
print("MSE:", round(tuned_xgb_train_mse, 3))
print("RMSE:", round(tuned_xgb_train_rmse, 3))
print("R²:", round(tuned_xgb_train_r2, 4))

Tuned XGBoost - Training
MAE: 37.484
MSE: 3148.902
RMSE: 56.115
R²: 0.9383


XGBoost Training and Validation Comparison

In [104]:
xgb_train_validation_comparison = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation"
    ],
    "MAE": [
        tuned_xgb_train_mae,
        best_xgb_result["MAE"]
    ],
    "MSE": [
        tuned_xgb_train_mse,
        best_xgb_result["MSE"]
    ],
    "RMSE": [
        tuned_xgb_train_rmse,
        best_xgb_result["RMSE"]
    ],
    "R2": [
        tuned_xgb_train_r2,
        best_xgb_result["R2"]
    ]
})

xgb_train_validation_comparison.round(4)

,Dataset,MAE,MSE,RMSE,R2
0,Training,37.4841,3148.9018,56.1151,0.9383
1,Validation,41.0639,3872.4644,62.2291,0.9199


Tune LightGBM

In [105]:
lgb_parameters = [
    {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 20,
        "max_depth": -1,
        "min_child_samples": 20,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": 8,
        "min_child_samples": 30,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 700,
        "learning_rate": 0.03,
        "num_leaves": 20,
        "max_depth": -1,
        "min_child_samples": 30,
        "colsample_bytree": 0.9
    },
    {
        "n_estimators": 400,
        "learning_rate": 0.08,
        "num_leaves": 15,
        "max_depth": 7,
        "min_child_samples": 30,
        "colsample_bytree": 0.8
    }
]

In [106]:
lgb_tuning_results = []

for i, params in enumerate(
    lgb_parameters,
    start=1
):
    print("Running experiment", i)

    lgb_tuning_model = LGBMRegressor(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        num_leaves=params["num_leaves"],
        max_depth=params["max_depth"],
        min_child_samples=params["min_child_samples"],
        colsample_bytree=params["colsample_bytree"],
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    lgb_tuning_model.fit(
        X_train,
        y_train
    )

    predictions = lgb_tuning_model.predict(
        X_validation
    )

    mae = mean_absolute_error(
        y_validation,
        predictions
    )

    mse = mean_squared_error(
        y_validation,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_validation,
        predictions
    )

    lgb_tuning_results.append({
        "Experiment": i,
        "Estimators": params["n_estimators"],
        "Learning Rate": params["learning_rate"],
        "Num Leaves": params["num_leaves"],
        "Max Depth": params["max_depth"],
        "Min Child Samples": params["min_child_samples"],
        "Colsample": params["colsample_bytree"],
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("RMSE:", round(rmse, 3))

Running experiment 1
RMSE: 62.338
Running experiment 2
RMSE: 63.861
Running experiment 3
RMSE: 64.42
Running experiment 4
RMSE: 63.218
Running experiment 5
RMSE: 63.976


Compare LightGBM Tuning Results

In [107]:
lgb_tuning_results_df = pd.DataFrame(
    lgb_tuning_results
)

lgb_tuning_results_df.sort_values(
    "RMSE"
).round(4)

,Experiment,Estimators,Learning Rate,Num Leaves,Max Depth,Min Child Samples,Colsample,MAE,MSE,RMSE,R2
0,1,500,0.03,31,-1,20,0.8,40.8152,3886.0195,62.3379,0.9196
3,4,700,0.03,20,-1,30,0.9,41.5540,3996.5584,63.2183,0.9173
1,2,500,0.05,20,-1,20,0.8,42.0856,4078.2617,63.8613,0.9156
4,5,400,0.08,15,7,30,0.8,42.5858,4092.9233,63.9760,0.9153
2,3,500,0.05,31,8,30,0.8,42.3872,4149.9930,64.4204,0.9142


In [110]:
best_lgb_result = (
    lgb_tuning_results_df
    .sort_values("RMSE")
    .iloc[0]
)

best_lgb_result

Experiment              1.000000
Estimators            500.000000
Learning Rate           0.030000
Num Leaves             31.000000
Max Depth              -1.000000
Min Child Samples      20.000000
Colsample               0.800000
MAE                    40.815248
MSE                  3886.019485
RMSE                   62.337946
R2                      0.919625
Name: 0, dtype: float64

Compare Initial and Tuned LightGBM

In [111]:
lgb_before_after = pd.DataFrame({
    "Model": [
        "Initial LightGBM",
        "Best Tuned LightGBM"
    ],
    "MAE": [
        lgb_mae,
        best_lgb_result["MAE"]
    ],
    "MSE": [
        lgb_mse,
        best_lgb_result["MSE"]
    ],
    "RMSE": [
        lgb_rmse,
        best_lgb_result["RMSE"]
    ],
    "R2": [
        lgb_r2,
        best_lgb_result["R2"]
    ]
})

lgb_before_after.round(4)

,Model,MAE,MSE,RMSE,R2
0,Initial LightGBM,42.4067,4203.8799,64.8373,0.9131
1,Best Tuned LightGBM,40.8152,3886.0195,62.3379,0.9196


Final Tuned LightGBM

In [113]:
tuned_lgb_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

tuned_lgb_model.fit(
    X_train,
    y_train
)

tuned_lgb_predictions = tuned_lgb_model.predict(
    X_validation
)

Tuned LightGBM Training Performance

In [114]:
tuned_lgb_train_predictions = tuned_lgb_model.predict(
    X_train
)

tuned_lgb_train_mae = mean_absolute_error(
    y_train,
    tuned_lgb_train_predictions
)

tuned_lgb_train_mse = mean_squared_error(
    y_train,
    tuned_lgb_train_predictions
)

tuned_lgb_train_rmse = np.sqrt(
    tuned_lgb_train_mse
)

tuned_lgb_train_r2 = r2_score(
    y_train,
    tuned_lgb_train_predictions
)

print("Tuned LightGBM - Training")
print("MAE:", round(tuned_lgb_train_mae, 3))
print("MSE:", round(tuned_lgb_train_mse, 3))
print("RMSE:", round(tuned_lgb_train_rmse, 3))
print("R²:", round(tuned_lgb_train_r2, 4))

Tuned LightGBM - Training
MAE: 38.573
MSE: 3264.262
RMSE: 57.134
R²: 0.936


LightGBM Training and Validation Comparison

In [115]:
lgb_train_validation_comparison = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation"
    ],
    "MAE": [
        tuned_lgb_train_mae,
        best_lgb_result["MAE"]
    ],
    "MSE": [
        tuned_lgb_train_mse,
        best_lgb_result["MSE"]
    ],
    "RMSE": [
        tuned_lgb_train_rmse,
        best_lgb_result["RMSE"]
    ],
    "R2": [
        tuned_lgb_train_r2,
        best_lgb_result["R2"]
    ]
})

lgb_train_validation_comparison.round(4)

,Dataset,MAE,MSE,RMSE,R2
0,Training,38.5733,3264.2619,57.1337,0.9360
1,Validation,40.8152,3886.0195,62.3379,0.9196


## Compare All Tuned Models

In [116]:
all_tuned_results = pd.DataFrame({
    "Model": [
        "Tuned Random Forest",
        "Tuned CatBoost",
        "Tuned XGBoost",
        "Tuned LightGBM"
    ],
    "MAE": [
        best_rf_result["MAE"],
        best_cat_result["MAE"],
        best_xgb_result["MAE"],
        best_lgb_result["MAE"]
    ],
    "MSE": [
        best_rf_result["MSE"],
        best_cat_result["MSE"],
        best_xgb_result["MSE"],
        best_lgb_result["MSE"]
    ],
    "RMSE": [
        best_rf_result["RMSE"],
        best_cat_result["RMSE"],
        best_xgb_result["RMSE"],
        best_lgb_result["RMSE"]
    ],
    "R2": [
        best_rf_result["R2"],
        best_cat_result["R2"],
        best_xgb_result["R2"],
        best_lgb_result["R2"]
    ]
})

all_tuned_results.sort_values(
    "RMSE"
).round(4)

,Model,MAE,MSE,RMSE,R2
0,Tuned Random Forest,38.2887,3544.4250,59.5351,0.9267
1,Tuned CatBoost,40.2052,3729.9972,61.0737,0.9229
2,Tuned XGBoost,41.0639,3872.4644,62.2291,0.9199
3,Tuned LightGBM,40.8152,3886.0195,62.3379,0.9196


## Stacking Ensemble

In [117]:
stack_times = (
    train_data["time_utc"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

stack_split = int(
    len(stack_times) * 0.80
)

stack_train_last_time = stack_times.iloc[
    stack_split - 1
]

stack_base_data = train_data[
    train_data["time_utc"] <= stack_train_last_time
].copy()

stack_meta_data = train_data[
    train_data["time_utc"] > stack_train_last_time
].copy()

print(
    "Base training rows:",
    len(stack_base_data)
)

print(
    "Meta training rows:",
    len(stack_meta_data)
)

print(
    "Base training ends:",
    stack_base_data["time_sl"].max()
)

print(
    "Meta training starts:",
    stack_meta_data["time_sl"].min()
)

Base training rows: 127430
Meta training rows: 31883
Base training ends: 2023-02-23 13:30:00+05:30
Meta training starts: 2023-02-23 14:30:00+05:30


In [118]:
X_stack_base = stack_base_data[
    feature_columns
]

y_stack_base = stack_base_data[
    target_column
]

X_stack_meta = stack_meta_data[
    feature_columns
]

y_stack_meta = stack_meta_data[
    target_column
]

Train Stacking Base Models

In [119]:
stack_rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")
stack_rf_model.fit(
    X_stack_base,
    y_stack_base
)


stack_cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

print("Training CatBoost...")
stack_cat_model.fit(
    X_stack_base,
    y_stack_base
)


stack_xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")
stack_xgb_model.fit(
    X_stack_base,
    y_stack_base
)


stack_lgb_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print("Training LightGBM...")
stack_lgb_model.fit(
    X_stack_base,
    y_stack_base
)

print("Base models completed.")

Training Random Forest...
Training CatBoost...
Training XGBoost...
Training LightGBM...
Base models completed.


Create Meta-Model Training Data

In [120]:
stack_rf_meta_predictions = stack_rf_model.predict(
    X_stack_meta
)

stack_cat_meta_predictions = stack_cat_model.predict(
    X_stack_meta
)

stack_xgb_meta_predictions = stack_xgb_model.predict(
    X_stack_meta
)

stack_lgb_meta_predictions = stack_lgb_model.predict(
    X_stack_meta
)

In [121]:
X_meta_train = pd.DataFrame({
    "Random Forest": stack_rf_meta_predictions,
    "CatBoost": stack_cat_meta_predictions,
    "XGBoost": stack_xgb_meta_predictions,
    "LightGBM": stack_lgb_meta_predictions
})

print(
    "Meta training shape:",
    X_meta_train.shape
)

X_meta_train.head()

Meta training shape: (31883, 4)


,Random Forest,CatBoost,XGBoost,LightGBM
0,476.171788,486.068984,492.930237,503.726141
1,545.488151,535.435067,548.557373,542.512453
2,514.714808,528.803463,524.592712,537.298061
3,567.495735,558.476199,573.343384,572.195986
4,592.333270,593.562695,600.559204,603.566649


 Train Stacking Meta-Model

In [123]:
stack_meta_model = LinearRegression()

stack_meta_model.fit(
    X_meta_train,
    y_stack_meta
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [125]:
stack_weights = pd.DataFrame({
    "Model": X_meta_train.columns,
    "Weight": stack_meta_model.coef_
})

stack_weights.round(4)

,Model,Weight
0,Random Forest,0.7583
1,CatBoost,0.3278
2,XGBoost,-0.1788
3,LightGBM,0.0948


In [126]:
print(
    "Intercept:",
    round(
        stack_meta_model.intercept_,
        4
    )
)

Intercept: -0.2924


Evaluate Stacking Ensemble

In [127]:
stack_rf_validation = stack_rf_model.predict(
    X_validation
)

stack_cat_validation = stack_cat_model.predict(
    X_validation
)

stack_xgb_validation = stack_xgb_model.predict(
    X_validation
)

stack_lgb_validation = stack_lgb_model.predict(
    X_validation
)

In [128]:
X_meta_validation = pd.DataFrame({
    "Random Forest": stack_rf_validation,
    "CatBoost": stack_cat_validation,
    "XGBoost": stack_xgb_validation,
    "LightGBM": stack_lgb_validation
})

In [129]:
stack_predictions = stack_meta_model.predict(
    X_meta_validation
)

In [130]:
stack_mae = mean_absolute_error(
    y_validation,
    stack_predictions
)

stack_mse = mean_squared_error(
    y_validation,
    stack_predictions
)

stack_rmse = np.sqrt(
    stack_mse
)

stack_r2 = r2_score(
    y_validation,
    stack_predictions
)

print("Stacking Ensemble")
print("MAE:", round(stack_mae, 3))
print("MSE:", round(stack_mse, 3))
print("RMSE:", round(stack_rmse, 3))
print("R²:", round(stack_r2, 4))

Stacking Ensemble
MAE: 38.704
MSE: 3587.152
RMSE: 59.893
R²: 0.9258


Stacking Prediction Check

In [131]:
print(
    "Negative predictions:",
    (stack_predictions < 0).sum()
)

print(
    "Minimum prediction:",
    stack_predictions.min()
)

print(
    "Maximum prediction:",
    stack_predictions.max()
)

Negative predictions: 1100
Minimum prediction: -4.6859636312627
Maximum prediction: 693.5085257386213


Final Validation Model Comparison

In [132]:
final_validation_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Tuned Random Forest",
        "Tuned CatBoost",
        "Tuned XGBoost",
        "Tuned LightGBM",
        "Stacking Ensemble"
    ],
    "MAE": [
        linear_mae,
        best_rf_result["MAE"],
        best_cat_result["MAE"],
        best_xgb_result["MAE"],
        best_lgb_result["MAE"],
        stack_mae
    ],
    "MSE": [
        linear_mse,
        best_rf_result["MSE"],
        best_cat_result["MSE"],
        best_xgb_result["MSE"],
        best_lgb_result["MSE"],
        stack_mse
    ],
    "RMSE": [
        linear_rmse,
        best_rf_result["RMSE"],
        best_cat_result["RMSE"],
        best_xgb_result["RMSE"],
        best_lgb_result["RMSE"],
        stack_rmse
    ],
    "R2": [
        linear_r2,
        best_rf_result["R2"],
        best_cat_result["R2"],
        best_xgb_result["R2"],
        best_lgb_result["R2"],
        stack_r2
    ]
})

final_validation_results.sort_values(
    "RMSE"
).round(4)

,Model,MAE,MSE,RMSE,R2
1,Tuned Random Forest,38.2887,3544.4250,59.5351,0.9267
5,Stacking Ensemble,38.7038,3587.1519,59.8928,0.9258
2,Tuned CatBoost,40.2052,3729.9972,61.0737,0.9229
3,Tuned XGBoost,41.0639,3872.4644,62.2291,0.9199
4,Tuned LightGBM,40.8152,3886.0195,62.3379,0.9196
0,Linear Regression,45.9533,4258.4467,65.2568,0.9119
